# MBB beam — fibre orientation to G-code

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Naruki-Ichihara/path_optimizer/blob/main/examples/mbb_fibre.ipynb)

Topology *and* fibre orientation optimised together, then carried through to a
printable toolpath:

```
optimise (density + orientation) -> aligned-SH stripes -> paths
    -> order -> connect -> FullControl design -> G-code
```

Runs on Colab. Set the parameters in the second cell; every later cell plots
what it produced.

In [ ]:
!pip install feax[cuda13,jax]
!pip install spineax
# nbformat is plotly's: FullControl's preview is a plotly figure, and showing
# one in a notebook fails with "Mime type rendering requires nbformat>=4.2.0"
# without it.  Colab ships it; a local Jupyter or VS Code kernel may not.
!pip install -q "nbformat>=4.2.0"
!pip install -q "path_optimizer @ git+https://github.com/Naruki-Ichihara/path_optimizer.git"


In [ ]:
import os

import feax as fe
import jax

print(fe.__version__)
print(jax.__version__)

## Parameters

In [ ]:
# Geometry and load (SI: metres, newtons)
LX, LY = 0.200, 0.050        # full span x height
NX, NY = 200, 50             # optimisation elements
THICKNESS = 2.0e-3
LOAD = 1.0e3                 # total downward force

# Materials.  No defaults anywhere in `materials` -- a stiffness nobody
# chose decides the optimum in silence.  Replace with yours.
E1, E2 = 140.0e9, 10.0e9     # along / across the fibre, Pa
G12, NU12 = 5.0e9, 0.30      # in-plane shear, major Poisson
# What fills the low-density regions.  None -> void: the design says WHERE THE
# PART IS and rho = 0 carries no load.  A Polymer instead makes it a two-phase
# blend -- the design then says how much fibre to add to a part that exists
# everywhere, and the "void" is as stiff as the matrix (1/59 of solid here).
MATRIX = None                # or materials.Polymer(E=2.0e9, nu=0.40)
E_MIN = 1e-9                 # void stiffness floor

# Optimisation
VOLUME_FRACTION = 0.4
FILTER_RADIUS = 3.0 * (LX / NX)
MAX_ITER = 150
# Sharpness of the off-diagonal (x3) projection.  2.0, not feax's own 10:
# at 10 it is saturated from the first iteration and the fibre comes out
# transverse to the load.  See materials.DEFAULT_SGN_BETA.
SGN_BETA = 2.0
W_UD = 0.5                   # rank-1 penalty weight
W_MAG = 0.5                  # |T| = rho penalty weight
W_SMOOTH = 0.0               # orientation smoothness weight

# Stripes.  Paths are the ZERO contours of the field and cos crosses zero twice
# per period, so the stripe period is twice the pitch you want to print at.
PATH_PITCH = 1.0e-3
STRIPE_PERIOD = 2.0 * PATH_PITCH
SH_NX, SH_NY = 600, 150      # ~6 elements per stripe period
SH_PHASE_STEPS = 10
SH_GATE_LO, SH_GATE_HI = 5.0, 15.0
SH_FREE_STEPS = 250
GAMMA_FREE = 8
RHO_CUTOFF = 0.5

# Toolpath.  Travels shorter than CONNECT_TOLERANCE are welded into bridges.
CONNECT_TOLERANCE = 5.0e-3
# Weld tolerance for the second pass, after the hairpins are trimmed out.
# Looser than the first: trimming leaves ends the original pass had already
# welded past, so a 5 mm reach rejoins fewer of them.
REJOIN_TOLERANCE = 10.0e-3
# Vertex thinning before the CAD files: 0.05 mm keeps ~5% of the vertices
# and leaves the path spacing unchanged.  Set to 0 to keep every vertex.
CAD_TOLERANCE = 0.05e-3
# The bead solid (`beads.step`): its cross-section, and how tall to make it.
# One 0.2 mm layer is invisible beside a 1 m beam, so the solid is written at
# the thickness the part is meant to be, as if the layers were stacked.
SOLID_HEIGHT = 20.0          # mm
# Tessellation tolerance for the viewer.  Flat faces are exact whatever it is;
# this cuts the round ends.  Loosen it if the viewer is slow to steer.
VIEW_DEFLECTION = 0.3        # mm

# Print
N_LAYERS = 4                 # how many layers to stack
LAYER_HEIGHT = 0.2           # mm
EXTRUSION_WIDTH = 0.4        # mm
PRINT_SPEED = 1000           # mm/min
RETRACT = True
Z_HOP = 0.6                  # mm
PRINTER = "prusa_mk4"        # a FullControl profile; "generic" has no ending procedure
PRINTER_SETTINGS = {"nozzle_temp": 240, "bed_temp": 60}

## Setup

In [ ]:
import feax.gene as gene
import fullcontrol as fc
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as onp

import path_optimizer as po
from path_optimizer import materials, paths, plane, stripes
from path_optimizer import objectives as obj

LAMINA = materials.Lamina(E1=E1, E2=E2, G12=G12, nu12=NU12)
FIELDS = (("rho_phys", "x1", "x2", "x3"),)


def show(title, aspect=True):
    fig, ax = plt.subplots(figsize=(14, 14 * LY / LX + 0.8))
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
    if aspect:
        ax.set_aspect("equal")
    return fig, ax

## Problem

In [ ]:
class MBBFibre(po.Pipeline):
    has_aux = True

    def build(self, mesh):
        pad = 2.0 * (LX / NX)
        pin = lambda p: (p[0] < pad) & (p[1] < 1e-9)
        roller = lambda p: (p[0] > LX - pad) & (p[1] < 1e-9)
        load = lambda p: (jnp.abs(p[0] - 0.5 * LX) < pad) & jnp.isclose(p[1], LY, atol=1e-9)

        self.problem = plane.make_plane_stress(
            mesh, materials.orientation_blend(LAMINA, MATRIX, sgn_beta=SGN_BETA,
                                        e_min=E_MIN),
            thickness=THICKNESS, location_fns=(load,),
            traction=(0.0, LOAD / (2.0 * pad)))   # residual side: +ty pulls down
        self.bc = fe.DirichletBCConfig([
            fe.DirichletBCSpec(location=pin, component=0, value=0.0),
            fe.DirichletBCSpec(location=pin, component=1, value=0.0),
            fe.DirichletBCSpec(location=roller, component=1, value=0.0),
        ]).create_bc(self.problem)

        nv = fe.TracedParams.create_node_var
        sample = fe.TracedParams(volume_vars=tuple(
            nv(self.problem, v) for v in (0.5, 0.0, 0.0, 0.0)))
        self.solver = po.make_linear_solver(self.problem, self.bc, sample)
        self.compliance = obj.create_compliance_fn(self.problem)
        self.volume = obj.create_volume_fn(self.problem)
        self.smoothness = obj.create_orientation_smoothness_fn(
            self.problem, length_scale=PATH_PITCH, sgn_beta=SGN_BETA)

        n = mesh.points.shape[0]
        start = fe.TracedParams(volume_vars=(
            jnp.full(n, VOLUME_FRACTION), jnp.zeros(n), jnp.zeros(n), jnp.zeros(n)))
        self.c0 = float(self.compliance(self.solver(start)))

    def transform(self, design, penalty=3.0, beta=1.0, **params):
        rho = gene.heaviside_projection(design["rho"], beta=beta)
        return {**design, "rho_phys": rho, "simp": rho ** penalty}

    def objective(self, design, **params):
        sol = self.solver(fe.TracedParams(volume_vars=(
            design["simp"], design["x1"], design["x2"], design["x3"])))
        c = self.compliance(sol) / self.c0
        ud = obj.ud_penalty(design, FIELDS, sgn_beta=SGN_BETA)
        mag = obj.magnitude_consistency(design, FIELDS, sgn_beta=SGN_BETA)
        smooth = self.smoothness(design, FIELDS)
        return (c + W_UD * ud + W_MAG * mag + W_SMOOTH * smooth,
                {"c": c, "ud": ud, "mag": mag, "smooth": smooth})

    @po.constraint(target=VOLUME_FRACTION)
    def volfrac(self, design, **params):
        return self.volume(design["rho_phys"])

## Optimise

In [ ]:
mesh = fe.mesh.rectangle_mesh(Nx=NX, Ny=NY, domain_x=LX, domain_y=LY, ele_type="QUAD4")
space = plane.design_space(
    oriented=True, rho_bounds=(1e-3, 1.0), rho_init=VOLUME_FRACTION,
    rho_filter_radius=FILTER_RADIUS, theta_filter_radius=FILTER_RADIUS)

result = po.run(
    MBBFibre(), mesh, space, max_iter=MAX_ITER,
    continuations={"penalty": po.Continuation(1.0, 3.0, update_every=40, step=0.5),
                   "beta": po.Continuation(1.0, 8.0, update_every=40, step=2.0)},
    write_xdmf=False)

print({k: round(float(v), 4) for k, v in result.final_aux.items()},
      "V/V0", round(result.final_constraints["volfrac"], 4))

In [ ]:
h = result.history
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
axs[0].semilogy(h["iter"], h["obj"], "b-")
axs[0].set_title("Objective"); axs[0].set_ylabel("objective")
axs[1].plot(h["iter"], h["volfrac"], label="volfrac")
axs[1].set_title("Constraints"); axs[1].set_ylabel("value")
for k in ("c", "ud", "mag", "smooth"):
    axs[2].plot(h["iter"], h[k], label=k)
axs[2].set_title("Diagnostics"); axs[2].set_ylabel("value")
for a in axs:
    a.set_xlabel("Iteration"); a.grid(alpha=0.3)
axs[1].legend(); axs[2].legend()
fig.tight_layout(); plt.show()

## Design

In [ ]:
pts = onp.asarray(result.mesh.points)
rho = onp.asarray(result.design["rho_phys"])
director = stripes.director_from_orientation(
    result.design["x1"], result.design["x2"], result.design["x3"], sgn_beta=SGN_BETA)

fig, ax = show(f"density + fibre director (compliance {result.final_aux['c']:.4g})")
ax.tripcolor(pts[:, 0], pts[:, 1], rho, cmap="gray_r", shading="gouraud", vmin=0, vmax=1)
sel = onp.where(rho > RHO_CUTOFF)[0][::6]
ax.quiver(pts[sel, 0], pts[sel, 1], director[sel, 0], director[sel, 1], color="tab:red",
          pivot="mid", headwidth=0, headlength=0, headaxislength=0, scale=60, width=0.0015)
plt.show()

## Stripes

In [ ]:
fine = fe.mesh.rectangle_mesh(Nx=SH_NX, Ny=SH_NY, domain_x=LX, domain_y=LY, ele_type="QUAD4")
field = stripes.stripes_from_result(
    result, FIELDS, STRIPE_PERIOD, mesh=fine, rho_cutoff=RHO_CUTOFF,
    sgn_beta=SGN_BETA, seed_from="hybrid", phase_steps=SH_PHASE_STEPS,
    gate_lo=SH_GATE_LO, gate_hi=SH_GATE_HI, free_steps=SH_FREE_STEPS,
    gamma_free=GAMMA_FREE)[0]

fp = onp.asarray(field.mesh.points)
fig, ax = show(f"aligned-SH stripes, period {STRIPE_PERIOD * 1e3:g} mm, {field.steps} steps")
ax.tripcolor(fp[:, 0], fp[:, 1], field.stripe, cmap="gray_r", shading="gouraud", vmin=0, vmax=1)
plt.show()

## Paths

In [ ]:
raw = paths.tag(paths.extract_paths(field), "fibre")
print(f"{len(raw)} paths, {sum(p.length for p in raw) * 1e3:.0f} mm")

fig, ax = show(f"{len(raw)} extracted paths")
for p in raw:
    ax.plot(*(p.nodes * 1e3).T, lw=0.6, color="k")
plt.show()

## Order, then connect

In [ ]:
ordered = paths.order_paths(raw)
joined = paths.connect(ordered, tolerance=CONNECT_TOLERANCE)
final = paths.order_paths(joined)
print(f"{len(ordered)} -> {len(joined)} paths | "
      f"travel {paths.travel_distance(ordered) * 1e3:.0f} -> "
      f"{paths.travel_distance(final) * 1e3:.0f} mm")

# Colour is the print order, not identity -- there are more chains than any
# categorical palette has slots, so a sequential ramp plus a colorbar is the
# only honest reading.
cmap = plt.cm.viridis
fig, ax = show(f"{len(final)} chains, coloured by print order; travels in grey")
for a, b in zip(final, final[1:]):
    ax.plot(*(onp.array([a.end, b.start]) * 1e3).T, lw=0.7, color="0.65", zorder=1)
for k, p in enumerate(final):
    c = cmap(k / max(len(final) - 1, 1))
    ax.plot(*(p.nodes * 1e3).T, lw=1.0, color=c, zorder=2)
    ax.plot(*(p.start * 1e3), "o", ms=5, mfc=c, mec="k", mew=0.5, zorder=3)
fig.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(1, len(final))),
             ax=ax, label="print order", fraction=0.02, pad=0.01)
plt.show()

## CAD export


In [ ]:
thin = paths.simplify(final, tolerance=CAD_TOLERANCE)
for ext, write in (("dxf", paths.write_dxf), ("step", paths.write_step)):
    write(thin, f"mbb_fibre.{ext}")
    print(f"mbb_fibre.{ext}: {os.path.getsize(f'mbb_fibre.{ext}') / 1024:.0f} kB")
print(f"{sum(len(p.nodes) for p in thin)} of "
      f"{sum(len(p.nodes) for p in final)} vertices kept")

# One layer: every layer of the stack is this same curve at its own z.


## Solids


In [ ]:
from path_optimizer import solids

bead = EXTRUSION_WIDTH * 1e-3
# A turn tighter than the bead's half-width is not one the printer can make --
# it lays material over material -- and it is what makes the solid
# self-intersect.  After `connect`, never before: the gap trimming opens is one
# bead wide and CONNECT_TOLERANCE would weld it shut again.
beads = paths.order_paths(paths.trim_hairpins(thin, radius=0.5 * bead, extra=3))
beads = paths.spread_ends(beads, move=0.1 * bead)
beads = paths.order_paths(
    paths.connect(beads, tolerance=REJOIN_TOLERANCE, no_cross=True))
print(f"{len(thin)} -> {len(beads)} paths | travel "
      f"{paths.travel_distance(thin) * 1e3:.0f} -> "
      f"{paths.travel_distance(beads) * 1e3:.0f} mm")

report = solids.write_step_solid(
    beads, "mbb_fibre_beads.step", width=EXTRUSION_WIDTH, height=SOLID_HEIGHT)
print(report)


Rotate, pan and section it. The frames are rendered on the kernel and streamed, so raise `VIEW_DEFLECTION` if it is slow to steer, or pass `jupyter_backend="static"` for a still image.


In [ ]:
solids.plot(solids.bead_compound(
    beads, width=EXTRUSION_WIDTH, height=SOLID_HEIGHT)[0],
    deflection=VIEW_DEFLECTION, view="iso")


## FullControl design

In [ ]:
steps = paths.to_fullcontrol_layers(
    [final] * N_LAYERS, z0=LAYER_HEIGHT, layer_height=LAYER_HEIGHT,
    width=EXTRUSION_WIDTH, print_speed=PRINT_SPEED, retract=RETRACT, hop=Z_HOP)
print(f"{N_LAYERS} layers, {len(steps)} steps")

# style="line": tube mode builds a mesh per segment and 58k points of it
# will not render in a browser.
fc.transform(steps, "plot", fc.PlotControls(
    style="line", color_type="z_gradient", zoom=0.7))

## G-code

In [ ]:
gcode = fc.transform(steps, "gcode", fc.GcodeControls(
    printer_name=PRINTER, initialization_data=PRINTER_SETTINGS), show_tips=False)
open("mbb_fibre.gcode", "w").write(gcode)
print(f"{len(gcode.splitlines())} lines, {len(gcode) / 1024:.0f} kB")

In [ ]:
from google.colab import files

for f in ("mbb_fibre.gcode", "mbb_fibre.dxf", "mbb_fibre.step",
          "mbb_fibre_beads.step"):
    files.download(f)
